# ICARUS-X M2 — Kp Forecasting Pipeline
## BiGRU + Bahdanau Attention | MC Dropout | 8-Horizon Probabilistic Forecast

### ✅ KAGGLE-READY — Run top to bottom. No changes needed.

**Inputs needed:**
- Upload `ar_features.csv` from M1 as a Kaggle Dataset (add in the Data panel on the right)
- OR the notebook generates realistic stub AR features automatically if file not found

**Outputs downloaded at end:**
- `best.pt` — trained BiGRU model
- `scalers.pkl` — normalisation scalers
- `infer.py` — inference function for M5
- `rmse_comparison.csv` — evaluation results

**Expected runtime on Kaggle T4 GPU: ~25 minutes**

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Environment Setup
# ═══════════════════════════════════════════════════════════
import os, sys, warnings, re, glob, shutil, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import requests
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Folder structure ────────────────────────────────────────
for d in ['data', 'data/omni_raw', 'data/windows', 'models', 'src']:
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, 'src')

# ── Device ──────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# ── Global config ────────────────────────────────────────────
HORIZONS   = [3, 6, 9, 12, 15, 18, 21, 24]
SW_COLS    = ['Bz', 'Vsw', 'Np', 'Pdyn']
WINDOW_HRS = 24
AR_DIM     = 12
MAX_EPOCHS = 60
BATCH_SIZE = 512
LR         = 1e-3
PATIENCE   = 12

print('✓ Setup complete')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Load ar_features.csv from M1
# Auto-generates stub if file not found
# ═══════════════════════════════════════════════════════════

def find_ar_features():
    """Find ar_features.csv in standard Kaggle input paths."""
    candidates = [
        'data/ar_features.csv',
        '/kaggle/input/ar-features/ar_features.csv',
        '/kaggle/input/icarus-m1/ar_features.csv',
        '/kaggle/input/icarus-x-m1/ar_features.csv',
    ]
    # Also scan any dataset added to this notebook
    for csv in glob.glob('/kaggle/input/**/*.csv', recursive=True):
        if 'ar_features' in csv.lower():
            candidates.append(csv)
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

def generate_stub_ar_features(n=500, seed=42):
    """Generate realistic stub AR features for 2010-2017 period."""
    rng = np.random.default_rng(seed)
    timestamps = pd.date_range('2013-01-01', '2016-12-31', periods=n)
    rows = []
    for i, ts in enumerate(timestamps):
        rows.append({
            'filename': ts.strftime('%Y-%m-%dT%H%M%S__magnetogram.jpg'),
            'n_regions_detected': int(rng.integers(0, 4)),
            **{f'f{j}': float(rng.uniform(0, 2)) for j in range(AR_DIM)}
        })
    df = pd.DataFrame(rows)
    df.to_csv('data/ar_features.csv', index=False)
    print(f'⚠  STUB: generated {n} synthetic AR feature rows')
    print('   To use real M1 output: add ar_features.csv as a Kaggle Dataset')
    return 'data/ar_features.csv'

ar_path = find_ar_features()
if ar_path:
    shutil.copy(ar_path, 'data/ar_features.csv')
    print(f'✓ Found ar_features.csv: {ar_path}')
    ar_df_raw = pd.read_csv('data/ar_features.csv')
    print(f'  Shape: {ar_df_raw.shape}')
    print(ar_df_raw.head(3))
else:
    ar_path = generate_stub_ar_features()

print('✓ AR features ready')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Download Kp Index (GFZ Potsdam)
# ═══════════════════════════════════════════════════════════

KP_CACHE = 'data/kp_1h.parquet'

if os.path.exists(KP_CACHE):
    kp_1h = pd.read_parquet(KP_CACHE)
    print(f'✓ Kp cached: {len(kp_1h):,} rows')
else:
    os.system('wget -q https://kp.gfz-potsdam.de/app/files/Kp_ap_Ap_SN_F107_since_1932.txt -O data/Kp_gfz.txt')

    kp_records = []
    with open('data/Kp_gfz.txt') as f:
        for line in f:
            if line.startswith('#') or not line.strip():
                continue
            parts = line.split()
            if len(parts) < 11:
                continue
            try:
                y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
                if not (2010 <= y <= 2020):
                    continue
                for i in range(8):
                    kp_val = float(parts[3 + i])
                    if kp_val > 9:
                        kp_val = kp_val / 10.0
                    if kp_val < 0 or kp_val > 9:
                        continue
                    ts = pd.Timestamp(y, m, d, i * 3)
                    kp_records.append({'timestamp': ts, 'Kp': kp_val})
            except:
                continue

    kp_df = pd.DataFrame(kp_records).set_index('timestamp')
    kp_1h = kp_df.resample('1h').ffill()
    kp_1h.to_parquet(KP_CACHE)

print(f'✓ Kp: {len(kp_1h):,} rows | range {kp_1h.index.min()} → {kp_1h.index.max()}')
print(f'  min={kp_1h.Kp.min():.1f}  max={kp_1h.Kp.max():.1f}  mean={kp_1h.Kp.mean():.2f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Download OMNI Solar Wind (2010-2020)
# Uses 1-hour low-res OMNI (faster + reliable on Kaggle)
# ═══════════════════════════════════════════════════════════

OMNI_CACHE = 'data/omni_kp_merged.parquet'

if os.path.exists(OMNI_CACHE):
    merged = pd.read_parquet(OMNI_CACHE)
    print(f'✓ OMNI cached: {len(merged):,} rows')
else:
    # OMNI2 hourly ASCII format column positions (0-indexed)
    # Col 39 = BZ_GSM (nT), 24 = V_SW (km/s), 23 = N_P (cm-3), 28 = Flow_pressure (nPa)
    # Ref: https://omniweb.gsfc.nasa.gov/html/ow_data.html
    FILL_VALS = {39: 9999.99, 24: 9999.0, 23: 999.9, 28: 99.99}

    records = []
    base = 'https://spdf.gsfc.nasa.gov/pub/data/omni/low_res_omni/'

    for year in range(2010, 2021):
        fname = f'omni2_{year}.dat'
        local = f'data/omni_raw/{fname}'

        if not os.path.exists(local):
            try:
                r = requests.get(base + fname, timeout=120)
                if r.status_code == 200:
                    with open(local, 'wb') as f:
                        f.write(r.content)
                else:
                    print(f'✗ {year}: HTTP {r.status_code}')
                    continue
            except Exception as e:
                print(f'✗ {year}: {e}')
                continue

        with open(local) as f:
            for line in f:
                parts = line.split()
                if len(parts) < 55:
                    continue
                try:
                    yr  = int(parts[0])
                    doy = int(parts[1])
                    hr  = int(parts[2])
                    ts  = pd.Timestamp(yr, 1, 1) + pd.Timedelta(days=doy-1, hours=hr)

                    bz   = float(parts[40])   # BZ_GSM — col 40 (1-indexed 41)
                    vsw  = float(parts[24])   # plasma speed
                    np_  = float(parts[23])   # proton density
                    pdyn = float(parts[28])   # flow pressure

                    # Replace fill values
                    if abs(bz)  > 999  : bz   = np.nan
                    if abs(vsw) > 9000 : vsw  = np.nan
                    if abs(np_) > 900  : np_  = np.nan
                    if abs(pdyn)> 90   : pdyn = np.nan

                    records.append({'timestamp': ts,
                                    'Bz': bz, 'Vsw': vsw,
                                    'Np': np_, 'Pdyn': pdyn})
                except:
                    continue

        print(f'  ✓ Parsed {year}: {len(records):,} total rows')

    omni = (pd.DataFrame(records)
              .set_index('timestamp')
              .sort_index())

    # Merge with Kp
    merged = omni.join(kp_1h, how='left')
    # Forward-fill Kp (3-hourly → hourly)
    merged['Kp'] = merged['Kp'].ffill()
    merged.to_parquet(OMNI_CACHE)
    print(f'✓ Saved merged: {merged.shape}')

print(f'✓ Merged shape : {merged.shape}')
print(f'  NaN %: {merged.isna().mean().round(3).to_dict()}')
merged.describe().round(2)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Normalise Solar Wind Features
# ═══════════════════════════════════════════════════════════

def build_and_apply_scalers(df, train_end='2017-12-31'):
    """Z-score normalise SW features using training period stats only."""
    train = df.loc[:train_end]
    scalers = {}
    df_norm = df.copy()
    for col in SW_COLS:
        mu  = train[col].mean()
        std = train[col].std()
        if std < 1e-8:
            std = 1.0
        scalers[col] = {'mean': float(mu), 'std': float(std)}
        df_norm[col] = (df[col] - mu) / std
    return df_norm, scalers

NORM_CACHE = 'data/omni_kp_normalised.parquet'

if os.path.exists(NORM_CACHE) and os.path.exists('models/scalers.pkl'):
    df_norm  = pd.read_parquet(NORM_CACHE)
    scalers  = joblib.load('models/scalers.pkl')
    print('✓ Loaded cached normalised data + scalers')
else:
    df_norm, scalers = build_and_apply_scalers(merged)
    df_norm.to_parquet(NORM_CACHE)
    joblib.dump(scalers, 'models/scalers.pkl')
    print('✓ Normalised + saved scalers')

print(f'\nScalers:')
for col, s in scalers.items():
    print(f'  {col:6s}: mean={s["mean"]:+.3f}  std={s["std"]:.3f}')

print(f'\nNormalised stats:')
print(df_norm[SW_COLS].describe().round(3))

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Generate Sliding-Window Dataset
# X_sw [N, 24, 4]  X_ar [N, 12]  y [N, 8]
# ═══════════════════════════════════════════════════════════

def parse_ar_timestamp(filename):
    """Extract timestamp from magnetogram filename."""
    m = re.search(r'(\d{4}-\d{2}-\d{2})T(\d{2})(\d{2})(\d{2})', str(filename))
    if m:
        date, hh, mm, ss = m.group(1), m.group(2), m.group(3), m.group(4)
        return pd.Timestamp(f'{date} {hh}:{mm}:{ss}')
    # Also try YYYY-MM-DD format
    m2 = re.search(r'(\d{4})[-_](\d{2})[-_](\d{2})', str(filename))
    if m2:
        return pd.Timestamp(f'{m2.group(1)}-{m2.group(2)}-{m2.group(3)}')
    return None

def load_ar_features(path='data/ar_features.csv'):
    """Load and index AR feature CSV by timestamp."""
    df = pd.read_csv(path)
    df['timestamp'] = df['filename'].apply(parse_ar_timestamp)
    df = df.dropna(subset=['timestamp']).set_index('timestamp').sort_index()
    df = df[~df.index.duplicated(keep='first')]
    feat_cols = [f'f{i}' for i in range(AR_DIM) if f'f{i}' in df.columns]
    # Pad to AR_DIM if fewer columns
    for j in range(len(feat_cols), AR_DIM):
        df[f'f{j}'] = 0.0
    return df[[f'f{i}' for i in range(AR_DIM)]]

SPLITS = {
    'train': ('2010-01-01', '2017-12-31'),
    'val':   ('2018-01-01', '2019-12-31'),
    'test':  ('2020-01-01', '2020-12-31'),
}
MAX_GAP_HRS = 3

WIN_CACHE = 'data/windows/X_sw_train.npy'

if os.path.exists(WIN_CACHE):
    print('✓ Windows cached — loading...')
else:
    ar_feat_df = load_ar_features('data/ar_features.csv')
    print(f'AR features: {ar_feat_df.shape}, range: {ar_feat_df.index.min()} → {ar_feat_df.index.max()}')

    dummy_ar = np.zeros(AR_DIM, dtype=np.float32)
    df_sw    = df_norm.sort_index()

    for split, (start, end) in SPLITS.items():
        print(f'\n{split.upper()} ({start} → {end})')
        times = df_sw.loc[start:end].index
        X_sw_list, X_ar_list, y_list = [], [], []
        skipped = 0

        for t in times:
            # ── Input window: 24 hours of SW ending at t ──────────────
            t0  = t - pd.Timedelta(hours=WINDOW_HRS)
            win = df_sw.loc[t0 : t - pd.Timedelta(hours=1), SW_COLS]

            if len(win) != WINDOW_HRS:
                skipped += 1
                continue

            sw = win.values.astype(np.float32)
            nan_count = np.isnan(sw).any(axis=1).sum()
            if nan_count > MAX_GAP_HRS:
                skipped += 1
                continue

            # Fill remaining NaN with column mean
            col_means = np.nanmean(sw, axis=0)
            for c in range(4):
                mask = np.isnan(sw[:, c])
                sw[mask, c] = col_means[c] if not np.isnan(col_means[c]) else 0.0

            # ── Targets: Kp at 8 future horizons ──────────────────────
            y = []
            valid = True
            for h in HORIZONS:
                tf = t + pd.Timedelta(hours=h)
                if tf not in df_sw.index:
                    valid = False
                    break
                kp_val = df_sw.loc[tf, 'Kp']
                if pd.isna(kp_val):
                    valid = False
                    break
                y.append(float(kp_val))
            if not valid:
                skipped += 1
                continue

            # ── AR features: nearest within 30 min ────────────────────
            try:
                idx  = ar_feat_df.index.get_indexer([t], method='nearest')[0]
                diff = abs((ar_feat_df.index[idx] - t).total_seconds()) / 60
                x_ar = ar_feat_df.iloc[idx].values.astype(np.float32) if diff <= 30 else dummy_ar
            except:
                x_ar = dummy_ar

            X_sw_list.append(sw)
            X_ar_list.append(x_ar)
            y_list.append(np.array(y, dtype=np.float32))

        X_sw = np.stack(X_sw_list)     # [N, 24, 4]
        X_ar = np.stack(X_ar_list)     # [N, 12]
        y_np = np.stack(y_list)        # [N, 8]

        np.save(f'data/windows/X_sw_{split}.npy', X_sw)
        np.save(f'data/windows/X_ar_{split}.npy', X_ar)
        np.save(f'data/windows/y_{split}.npy',    y_np)

        print(f'  Windows: {len(X_sw):,}  Skipped: {skipped:,}')
        print(f'  X_sw {X_sw.shape}  X_ar {X_ar.shape}  y {y_np.shape}')

    print('\n✓ All windows saved')

# Load and verify
for split in ['train', 'val', 'test']:
    X = np.load(f'data/windows/X_sw_{split}.npy')
    print(f'{split:5s}: {X.shape[0]:,} windows')

print('✓ Window generation complete')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Model Architecture
# BiGRU + Bahdanau Attention + AR Feature Fusion
# ═══════════════════════════════════════════════════════════

class BahdanauAttention(nn.Module):
    """Additive attention over BiGRU encoder hidden states."""

    def __init__(self, hidden_dim: int):
        super().__init__()
        self.W_h = nn.Linear(hidden_dim * 2, hidden_dim, bias=False)
        self.v   = nn.Linear(hidden_dim, 1,  bias=False)

    def forward(self, encoder_outputs: torch.Tensor):
        """encoder_outputs: [B, seq, hidden*2] → context [B, hidden*2], weights [B, seq]"""
        energy  = torch.tanh(self.W_h(encoder_outputs))   # [B, seq, hidden]
        scores  = self.v(energy).squeeze(-1)               # [B, seq]
        weights = torch.softmax(scores, dim=-1)            # [B, seq]
        context = torch.bmm(weights.unsqueeze(1),
                            encoder_outputs).squeeze(1)    # [B, hidden*2]
        return context, weights


class BiGRUPredictor(nn.Module):
    """
    BiGRU Seq2Seq + Bahdanau Attention + AR feature fusion.
    Outputs 8-horizon probabilistic Kp forecast via MC Dropout.
    """

    def __init__(self, sw_dim=4, ar_dim=12, hidden=256,
                 n_layers=2, dropout=0.15, n_horizons=8):
        super().__init__()
        self.gru = nn.GRU(
            input_size  = sw_dim,
            hidden_size = hidden,
            num_layers  = n_layers,
            batch_first = True,
            bidirectional = True,
            dropout = dropout if n_layers > 1 else 0.0
        )
        self.attention = BahdanauAttention(hidden)
        self.drop      = nn.Dropout(dropout)

        # Fusion: context [hidden*2] + AR [ar_dim] → hidden
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2 + ar_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(inplace=True)
        )
        self.decoder = nn.Linear(hidden, n_horizons)

    def forward(self, x_sw: torch.Tensor, x_ar: torch.Tensor):
        """x_sw [B,24,4]  x_ar [B,12] → predictions [B,8], attn_weights [B,24]"""
        enc, _    = self.gru(x_sw)                        # [B, 24, hidden*2]
        enc       = self.drop(enc)
        ctx, attn = self.attention(enc)                   # [B, hidden*2], [B, 24]
        fused     = self.drop(torch.cat([ctx, x_ar], -1)) # [B, hidden*2+ar_dim]
        out       = self.decoder(self.fusion(fused))      # [B, 8]
        return out, attn

    @torch.no_grad()
    def mc_dropout_predict(self, x_sw, x_ar, n_passes=50):
        """
        Monte Carlo Dropout inference — keeps dropout ACTIVE.
        Returns mean, std, p5, p95 [B, 8] and final attention [B, 24].
        """
        self.train()          # keeps dropout active
        preds = []
        for _ in range(n_passes):
            p, attn = self.forward(x_sw, x_ar)
            preds.append(p.unsqueeze(0))
        preds = torch.cat(preds, dim=0)   # [n_passes, B, 8]
        self.eval()
        _, attn_final = self.forward(x_sw, x_ar)
        return (preds.mean(0), preds.std(0),
                torch.quantile(preds, 0.05, dim=0),
                torch.quantile(preds, 0.95, dim=0),
                attn_final)


# Verify architecture
model_test = BiGRUPredictor().to(DEVICE)
x_sw_t = torch.zeros(2, 24, 4).to(DEVICE)
x_ar_t = torch.zeros(2, 12).to(DEVICE)
out_t, attn_t = model_test(x_sw_t, x_ar_t)
print(f'✓ Model forward pass: input [2,24,4] + [2,12] → output {tuple(out_t.shape)}')
print(f'  Attention weights:  {tuple(attn_t.shape)}')
n_params = sum(p.numel() for p in model_test.parameters())
print(f'  Parameters: {n_params:,}')
del model_test

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Training Loop
# ═══════════════════════════════════════════════════════════

def load_split(split, window_dir='data/windows'):
    """Load precomputed windows as tensors."""
    X_sw = torch.tensor(np.load(f'{window_dir}/X_sw_{split}.npy'))
    X_ar = torch.tensor(np.load(f'{window_dir}/X_ar_{split}.npy'))
    y    = torch.tensor(np.load(f'{window_dir}/y_{split}.npy'))
    return X_sw, X_ar, y


def persistence_rmse(y: torch.Tensor) -> torch.Tensor:
    """Baseline: predict the most recent Kp for all horizons."""
    # y[:, 0] is the 3-hr ahead Kp — use it as proxy for 'current' Kp
    pred = y[:, 0:1].expand_as(y)
    return torch.sqrt(((y - pred) ** 2).mean(dim=0))


# ── Load data ────────────────────────────────────────────────
X_sw_tr, X_ar_tr, y_tr = load_split('train')
X_sw_vl, X_ar_vl, y_vl = load_split('val')
print(f'Train: {X_sw_tr.shape[0]:,} windows')
print(f'Val  : {X_sw_vl.shape[0]:,} windows')

train_dl = DataLoader(TensorDataset(X_sw_tr, X_ar_tr, y_tr),
                      batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True)
val_dl   = DataLoader(TensorDataset(X_sw_vl, X_ar_vl, y_vl),
                      batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)

# ── Model + optimiser ────────────────────────────────────────
model   = BiGRUPredictor().to(DEVICE)
opt     = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5, verbose=False)
crit    = nn.MSELoss()

best_h3_rmse = float('inf')
no_improve   = 0
history      = []

print(f'\nStarting training: {MAX_EPOCHS} epochs max, early stop patience={PATIENCE}')
print('=' * 85)

for epoch in range(1, MAX_EPOCHS + 1):

    # ── Train ────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for xsw, xar, yb in train_dl:
        xsw, xar, yb = xsw.to(DEVICE), xar.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        pred, _ = model(xsw, xar)
        loss    = crit(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        train_loss += loss.item() * len(xsw)
    train_loss /= len(X_sw_tr)

    # ── Validate ─────────────────────────────────────────────
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for xsw, xar, yb in val_dl:
            p, _ = model(xsw.to(DEVICE), xar.to(DEVICE))
            all_pred.append(p.cpu())
            all_true.append(yb)
    all_pred = torch.cat(all_pred)
    all_true = torch.cat(all_true)
    val_rmse = torch.sqrt(((all_pred - all_true) ** 2).mean(dim=0))
    h3_rmse  = val_rmse[0].item()
    sched.step(h3_rmse)

    rmse_str = '  '.join([f'h{h}={r:.3f}' for h, r
                           in zip(HORIZONS, val_rmse.tolist())])
    star = ''
    if h3_rmse < best_h3_rmse:
        best_h3_rmse = h3_rmse
        no_improve   = 0
        torch.save({'model_state': model.state_dict(),
                    'epoch': epoch,
                    'val_rmse': val_rmse.tolist()},
                   'models/best.pt')
        star = '  ← best'
    else:
        no_improve += 1

    print(f'Ep {epoch:03d} | loss={train_loss:.4f} | {rmse_str}{star}')
    history.append({'epoch': epoch, 'train_loss': train_loss,
                    'h3_rmse': h3_rmse,
                    'h24_rmse': val_rmse[-1].item()})

    if no_improve >= PATIENCE:
        print(f'\nEarly stop at epoch {epoch} (no improve for {PATIENCE} epochs)')
        break

print('\n✓ Training complete')
print(f'  Best 3hr RMSE on val: {best_h3_rmse:.4f}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Training Curve Plot
# ═══════════════════════════════════════════════════════════

hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', color='steelblue')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss'); axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(hist_df['epoch'], hist_df['h3_rmse'],  label='3hr RMSE',  color='darkorange')
axes[1].plot(hist_df['epoch'], hist_df['h24_rmse'], label='24hr RMSE', color='darkred', linestyle='--')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('RMSE (Kp)')
axes[1].set_title('Validation RMSE by Horizon'); axes[1].grid(alpha=0.3); axes[1].legend()

plt.suptitle('ICARUS-X M2 — BiGRU Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('models/training_curve.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Saved: models/training_curve.png')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — Full Evaluation on Test Set (2020)
# RMSE table: Model+AR vs Model-AR vs Persistence
# ═══════════════════════════════════════════════════════════

# Load best checkpoint
ckpt = torch.load('models/best.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'✓ Loaded best checkpoint (epoch {ckpt["epoch"]})')

# Load test set
X_sw_te, X_ar_te, y_te = load_split('test')
test_dl = DataLoader(TensorDataset(X_sw_te, X_ar_te, y_te),
                     batch_size=512, shuffle=False, num_workers=2)
print(f'Test set: {X_sw_te.shape[0]:,} windows')

zeros_ar = torch.zeros_like(X_ar_te)

pred_ar_list, pred_noar_list, true_list = [], [], []

with torch.no_grad():
    for xsw, xar, yb in test_dl:
        xsw_d  = xsw.to(DEVICE)
        xar_d  = xar.to(DEVICE)
        zer_d  = torch.zeros_like(xar_d)

        p_ar,   _ = model(xsw_d, xar_d)
        p_noar, _ = model(xsw_d, zer_d)

        pred_ar_list.append(p_ar.cpu())
        pred_noar_list.append(p_noar.cpu())
        true_list.append(yb)

pred_ar   = torch.cat(pred_ar_list)
pred_noar = torch.cat(pred_noar_list)
true_all  = torch.cat(true_list)

rmse_ar   = torch.sqrt(((pred_ar   - true_all) ** 2).mean(dim=0))
rmse_noar = torch.sqrt(((pred_noar - true_all) ** 2).mean(dim=0))
rmse_pers = persistence_rmse(true_all)

# ── Print table ──────────────────────────────────────────────
print('\n' + '=' * 78)
print(f'{"Horizon":>8} {"Model+AR":>10} {"Model-AR":>10} {"Persist":>10} {"Δ% vs Persist":>15} {"AR Helps?": >10}')
print('=' * 78)

rows = []
for i, h in enumerate(HORIZONS):
    m_ar   = rmse_ar[i].item()
    m_noar = rmse_noar[i].item()
    p      = rmse_pers[i].item()
    # Safe division — avoid ZeroDivisionError
    delta_str = f'{(m_ar - p) / p * 100:+.1f}%' if p > 1e-8 else 'N/A'
    ar_helps  = 'YES ✓' if m_ar < m_noar else 'no'
    print(f'{h:>6}hr  {m_ar:>10.4f} {m_noar:>10.4f} {p:>10.4f} {delta_str:>15} {ar_helps:>10}')
    rows.append({'horizon_hr': h, 'model_rmse': round(m_ar, 4),
                 'no_ar_rmse': round(m_noar, 4),
                 'persistence_rmse': round(p, 4),
                 'delta_pct': delta_str})

print('=' * 78)
print(f'\nAblation (3hr): with AR={rmse_ar[0]:.4f}  without AR={rmse_noar[0]:.4f}')
print(f'M1 image features improve 3hr RMSE: {"YES" if rmse_ar[0] < rmse_noar[0] else "NO"}')

# Save results
rmse_df = pd.DataFrame(rows)
rmse_df.to_csv('models/rmse_comparison.csv', index=False)
print('\n✓ Saved: models/rmse_comparison.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — RMSE Plot (for report/presentation)
# ═══════════════════════════════════════════════════════════

x = np.arange(len(HORIZONS))
w = 0.28

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w,     rmse_ar.numpy(),   width=w, label='BiGRU + AR Features', color='#1565C0')
ax.bar(x,         rmse_noar.numpy(), width=w, label='BiGRU (no AR)',        color='#42A5F5')
ax.bar(x + w,     rmse_pers.numpy(), width=w, label='Persistence Baseline', color='#B0BEC5')
ax.set_xticks(x)
ax.set_xticklabels([f'{h}hr' for h in HORIZONS])
ax.set_xlabel('Forecast Horizon')
ax.set_ylabel('RMSE (Kp units)')
ax.set_title('ICARUS-X M2 — BiGRU Kp Forecast RMSE by Horizon (Test Set 2020)',
             fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(rmse_pers.max().item(), rmse_ar.max().item()) * 1.25)

plt.tight_layout()
plt.savefig('models/rmse_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Saved: models/rmse_comparison.png')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 12 — Write infer.py (for M5 handoff)
# This is a self-contained file — M5 imports run_forecast()
# ═══════════════════════════════════════════════════════════

INFER_CODE = '''
"""
ICARUS-X M2 — infer.py
Single-call inference for the M5 Architect module.
M5 calls: from infer import run_forecast
"""

import sys, os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import joblib
from datetime import datetime
from pathlib import Path

# ── Model definition (self-contained, no external import) ────────────────
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_h = nn.Linear(hidden_dim * 2, hidden_dim, bias=False)
        self.v   = nn.Linear(hidden_dim, 1,  bias=False)
    def forward(self, enc):
        energy  = torch.tanh(self.W_h(enc))
        scores  = self.v(energy).squeeze(-1)
        weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), enc).squeeze(1)
        return context, weights

class BiGRUPredictor(nn.Module):
    def __init__(self, sw_dim=4, ar_dim=12, hidden=256,
                 n_layers=2, dropout=0.15, n_horizons=8):
        super().__init__()
        self.gru = nn.GRU(sw_dim, hidden, n_layers, batch_first=True,
                          bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0.0)
        self.attention = BahdanauAttention(hidden)
        self.drop      = nn.Dropout(dropout)
        self.fusion    = nn.Sequential(
            nn.Linear(hidden * 2 + ar_dim, hidden),
            nn.LayerNorm(hidden), nn.ReLU(inplace=True)
        )
        self.decoder   = nn.Linear(hidden, n_horizons)
    def forward(self, x_sw, x_ar):
        enc, _ = self.gru(x_sw)
        enc    = self.drop(enc)
        ctx, attn = self.attention(enc)
        fused  = self.drop(torch.cat([ctx, x_ar], -1))
        return self.decoder(self.fusion(fused)), attn
    @torch.no_grad()
    def mc_dropout_predict(self, x_sw, x_ar, n_passes=50):
        self.train()
        preds = [self.forward(x_sw, x_ar)[0].unsqueeze(0) for _ in range(n_passes)]
        preds = torch.cat(preds, dim=0)
        self.eval()
        _, attn = self.forward(x_sw, x_ar)
        return (preds.mean(0), preds.std(0),
                torch.quantile(preds, 0.05, dim=0),
                torch.quantile(preds, 0.95, dim=0), attn)

# ── Constants ─────────────────────────────────────────────────────────────
HORIZONS = [3, 6, 9, 12, 15, 18, 21, 24]
SW_COLS  = [\'Bz\', \'Vsw\', \'Np\', \'Pdyn\']

# ── Main inference function ────────────────────────────────────────────────
def run_forecast(
    solar_wind_df,
    ar_features,
    ckpt_path   = \'models/best.pt\',
    scalers_path = \'models/scalers.pkl\',
    n_passes    = 50,
    device      = None
):
    """
    Main inference function — called by M5 model_runner.py

    Args:
        solar_wind_df : pd.DataFrame with columns [Bz,Vsw,Np,Pdyn], >= 24 rows
        ar_features   : list or np.ndarray of shape [12] — from M1
        ckpt_path     : path to best.pt checkpoint
        scalers_path  : path to scalers.pkl
        n_passes      : MC Dropout passes for uncertainty (default 50)
        device        : torch device (auto-detected if None)

    Returns:
        dict: run_timestamp + list of 8 horizon dicts
    """
    if device is None:
        device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')

    # ── Normalise solar wind ──────────────────────────────────────────────
    scalers = joblib.load(scalers_path)
    sw = solar_wind_df[SW_COLS].copy().astype(float)
    for col, s in scalers.items():
        sw[col] = (sw[col] - s[\'mean\']) / max(s[\'std\'], 1e-8)

    # Ensure exactly 24 rows
    if len(sw) < 24:
        pad = pd.DataFrame(np.zeros((24 - len(sw), 4)), columns=SW_COLS)
        sw  = pd.concat([pad, sw], ignore_index=True)
    sw_arr = sw.tail(24).values.astype(np.float32)
    sw_arr = np.nan_to_num(sw_arr, nan=0.0)

    # ── Build tensors ────────────────────────────────────────────────────
    x_sw = torch.tensor(sw_arr).unsqueeze(0).to(device)                   # [1, 24, 4]
    x_ar = torch.tensor(np.array(ar_features, dtype=np.float32)
                        ).unsqueeze(0).to(device)                          # [1, 12]

    # ── Load model ───────────────────────────────────────────────────────
    model = BiGRUPredictor().to(device)
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt[\'model_state\'])

    # ── MC Dropout inference ─────────────────────────────────────────────
    mean, std, p5, p95, attn = model.mc_dropout_predict(x_sw, x_ar, n_passes)

    mean = np.clip(mean.squeeze(0).cpu().numpy(), 0, 9)
    std  = std.squeeze(0).cpu().numpy()
    p5   = np.clip(p5.squeeze(0).cpu().numpy(),  0, 9)
    p95  = np.clip(p95.squeeze(0).cpu().numpy(), 0, 9)
    attn = attn.squeeze(0).cpu().numpy().tolist()  # [24] hourly attention weights

    return {
        \'run_timestamp\': datetime.utcnow().strftime(\' %Y-%m-%dT%H:%M:%S\').strip(),
        \'horizons\': [
            {
                \'horizon_hr\'       : int(HORIZONS[i]),
                \'kp_predicted\'    : round(float(mean[i]), 3),
                \'kp_ci_low\'       : round(float(p5[i]),   3),
                \'kp_ci_high\'      : round(float(p95[i]),  3),
                \'kp_std\'          : round(float(std[i]),  3),
                \'attention_weights\': attn,
            }
            for i in range(8)
        ]
    }


if __name__ == \'__main__\':
    import json
    demo_sw = pd.DataFrame({
        \'Bz\'  : np.full(24, -10.0),
        \'Vsw\' : np.full(24,  600.0),
        \'Np\'  : np.full(24,   10.0),
        \'Pdyn\': np.full(24,    2.5),
    })
    demo_ar = [0.1] * 12
    out = run_forecast(demo_sw, demo_ar)
    # Print without attention weights
    for h in out[\'horizons\']:
        h[\'attention_weights\'] = \'[... 24 floats ...]\'   # truncate for display
    print(json.dumps(out, indent=2))
'''

with open('src/infer.py', 'w') as f:
    f.write(INFER_CODE)

print('✓ Written: src/infer.py')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 13 — Test run_forecast() with Sept 2017 storm inputs
# ═══════════════════════════════════════════════════════════

import importlib.util, json

# Load infer.py without relying on sys.path
spec = importlib.util.spec_from_file_location('infer', 'src/infer.py')
infer_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(infer_mod)
run_forecast = infer_mod.run_forecast

# ── Sept 2017 X9.3 storm scenario ───────────────────────────
demo_sw = pd.DataFrame({
    'Bz':   np.full(24, -28.0),   # Very negative Bz = severe storm driver
    'Vsw':  np.full(24, 770.0),   # High solar wind speed
    'Np':   np.full(24,  15.0),   # Elevated proton density
    'Pdyn': np.full(24,   5.5),   # High dynamic pressure
})
demo_ar = [1.8, 0.15, 0.65, 0.42, 0.88, 0.55, 0.71, 1.3, 2.9, 0.79, 1.1, 0.82]

result = run_forecast(
    solar_wind_df = demo_sw,
    ar_features   = demo_ar,
    ckpt_path     = 'models/best.pt',
    scalers_path  = 'models/scalers.pkl',
    n_passes      = 20   # faster for demo; use 50 in production
)

print('\n━━━ ICARUS-X M2 — Sample Output (Sept 2017 Storm Scenario) ━━━')
print(f'Timestamp: {result["run_timestamp"]}')
print(f'{'Horizon':>8}  {'Kp Mean':>8}  {'CI [5–95%]':>14}  {'Std':>6}')
print('─' * 46)
for h in result['horizons']:
    print(f'{h["horizon_hr"]:>6}hr  '
          f'{h["kp_predicted"]:>8.3f}  '
          f'[{h["kp_ci_low"]:>5.2f} – {h["kp_ci_high"]:>5.2f}]  '
          f'{h["kp_std"]:>6.3f}')
print('─' * 46)
print('\n✓ run_forecast() works — output matches M5 contract format')
print('✓ Attention weights: list of 24 floats (hourly importance)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 14 — Attention Weight Visualisation
# ═══════════════════════════════════════════════════════════

attn_weights = np.array(result['horizons'][0]['attention_weights'])  # 24 values

fig, ax = plt.subplots(figsize=(12, 3))
bars = ax.bar(range(24), attn_weights, color='steelblue', alpha=0.85, edgecolor='white')
# Highlight top-3 most attended hours
top3 = np.argsort(attn_weights)[-3:]
for i in top3:
    bars[i].set_color('#E65100')

ax.set_xticks(range(24))
ax.set_xticklabels([f'-{24-i}h' if i < 24 else 'now' for i in range(24)], fontsize=8)
ax.set_xlabel('Hours Before Forecast', fontsize=11)
ax.set_ylabel('Attention Weight', fontsize=11)
ax.set_title('Bahdanau Attention — Which hours drove the forecast?\n'
             '(Orange = top 3 most attended hours)', fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('models/attention_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Saved: models/attention_heatmap.png')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 15 — Summary + Download All Deliverables
# ═══════════════════════════════════════════════════════════

import zipfile

print('=' * 60)
print('   ICARUS-X M2 — DELIVERABLES SUMMARY')
print('=' * 60)

deliverables = [
    ('models/best.pt',              'Trained BiGRU model checkpoint'),
    ('models/scalers.pkl',          'Normalisation scalers (hand to M5)'),
    ('src/infer.py',                'Inference function (hand to M5)'),
    ('models/rmse_comparison.csv',  'RMSE table vs persistence baseline'),
    ('models/training_curve.png',   'Training loss + RMSE curves'),
    ('models/rmse_comparison.png',  'RMSE bar chart by horizon'),
    ('models/attention_heatmap.png','Bahdanau attention visualisation'),
]

print()
all_ok = True
for path, desc in deliverables:
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1024:.1f} KB' if exists else '---'
    status = '✓' if exists else '✗'
    if not exists:
        all_ok = False
    print(f'  {status} {path:40s} {size:>10}   {desc}')

# Zip everything for easy download
ZIP_PATH = '/kaggle/working/icarus_x_m2_outputs.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path, _ in deliverables:
        if os.path.exists(path):
            zf.write(path)

zip_size = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f'\n✓ All outputs zipped: {ZIP_PATH} ({zip_size:.1f} MB)')

print()
print('=' * 60)
print('   WHAT TO HAND TO YOUR TEAM')
print('=' * 60)
print()
print('  → M5 (Architect) needs:')
print('    1. models/best.pt        (copy to icarus-x/models/)')
print('    2. models/scalers.pkl    (copy to icarus-x/models/)')
print('    3. src/infer.py          (copy to icarus-x/m2_predictor/)')
print()
print('  → For the project report, use:')
print('    4. models/rmse_comparison.csv   (RMSE table numbers)')
print('    5. models/rmse_comparison.png   (Figure for report)')
print('    6. models/training_curve.png    (Training evidence)')
print()
print('  Download: /kaggle/working/icarus_x_m2_outputs.zip')
print('  In Kaggle: Output tab → download the zip file')
print()
print('  ─── FINAL RESULTS ───')
rmse_final = pd.read_csv('models/rmse_comparison.csv')
print(rmse_final.to_string(index=False))
print()
print('✅ M2 COMPLETE' if all_ok else '⚠  Some files missing — check above')